# 00 · Setup, Databricks assets and compute

**Scenario:** Choose supported compute, inspect the source and understand the course write boundaries.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Inspect schema and data quality

Inspect the original types before transformations. Do not install a separate PySpark over the Databricks-provided package.


In [ ]:
import sys
print('Python:',sys.version)
print('Spark:',spark.version)
asset.printSchema()
profile=df.agg(F.count('*').alias('rows'),F.countDistinct('trip_id').alias('distinct_fingerprints'),
    F.min('pickup_ts').alias('first_pickup'),F.max('pickup_ts').alias('last_pickup'))
profile.show(truncate=False)
assert df.count()<=MAX_ROWS


**Expected check:** At least one asset row is available and the sample is bounded.


## Step 3 — Browse Databricks file assets optionally

The Unity Catalog sample volume is a second official asset location. Listing it is optional; this course still reads only the taxi table for record data.


In [ ]:
BROWSE_SAMPLE_VOLUME=False
if BROWSE_SAMPLE_VOLUME:
    display(dbutils.fs.ls('/Volumes/samples/databricks/datasets/'))
else:
    print('Optional sample-volume listing skipped.')
print('DataFrame labs: recent serverless or classic compute.')
print('RDD and DB2 partition labs: dedicated classic compute; explicit switches are off.')
print('Delta writes: writable catalog/schema. File/streaming labs: writable UC volume.')


**Expected check:** Default execution performs no optional listing or resource creation.


## Practice

Open notebook 01. Set max_rows to 500 and rerun, then inspect the schema and date range.

## Solution guide

The row limit changes the learning slice. Query plans can still scan/sort the larger source; use proper source predicates for production cost control.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
